In [1]:
import os
import random
import warnings
from collections import defaultdict
from itertools import combinations

import pandas as pd
import plotly.graph_objects as go
import torch
from joblib import Parallel, delayed
from pandas import DataFrame, read_csv
from pted import pted
from scipy.stats import mannwhitneyu, wasserstein_distance_nd
from skbio.stats.distance import DistanceMatrix, permanova, permdisp
from tqdm.auto import tqdm

from figure_style import FONT, GREEN, TICK, TITLE, TMLR_DIR, write_tmlr_dropdown_html, write_tmlr_html
from residual_stream import output_folder_name
from utils import centroid, cos_dist, euc, gram_condition_number, normalize

TEXT_FOLDER = 'core_clinical_short/'
MODELS = [
    'google/gemma-3-27b-pt',
    'google/gemma-3-27b-it',
    'google/gemma-3-12b-pt',
    'google/gemma-3-12b-it',
    'google/gemma-3-4b-pt',
    'google/gemma-3-4b-it',
    'mistralai/Mistral-Small-24B-Base-2501',
    'mistralai/Mistral-Small-24B-Instruct-2501',
    'allenai/OLMo-2-0325-32B',
    'allenai/OLMo-2-0325-32B-Instruct',
]
DIST_TYPES = ['centroid_cosine', 'centroid_cosine_unit',
              'centroid_euclidean', 'centroid_euclidean_unit',
              'earth_mover', 'earth_mover_unit',
              'energy', 'energy_unit']
PERMU = 9999
ALPHA = 0.05  # PERMDISP gate: a layer is eligible when PERMDISP p > ALPHA
SEED = 42  # permutation seed for PERMANOVA / PERMDISP, so p-values are reproducible
N_JOBS = max(1, os.cpu_count() - 1)  # joblib workers: all CPU cores but one

PERMANOVA_CSV = '82.permanova.csv'
PERMDISP_CSV = '82.permdisp.csv'
PEAKS_CSV = '82.gated_peaks.csv'
OPERATING_CSV = '82.operating_layers.csv'  # per-model operating layer, for the cross-model analysis


def model_short(m):
    return m.split('/')[-1]


# Compact figure labels (phone width): PT / IT for every base / instruction-tuned pair, and
# Mistral's "Small" dropped. The dropped suffixes are release dates: Mistral 2501 = YYMM
# (January 2025), OLMo 0325 = MMYY (March 2025). model_short keeps the full name for CSV
# filenames and keys.
DISPLAY = {
    'gemma-3-27b-pt': 'Gemma-3-27B-PT',
    'gemma-3-27b-it': 'Gemma-3-27B-IT',
    'gemma-3-12b-pt': 'Gemma-3-12B-PT',
    'gemma-3-12b-it': 'Gemma-3-12B-IT',
    'gemma-3-4b-pt': 'Gemma-3-4B-PT',
    'gemma-3-4b-it': 'Gemma-3-4B-IT',
    'Mistral-Small-24B-Base-2501': 'Mistral-24B-PT',
    'Mistral-Small-24B-Instruct-2501': 'Mistral-24B-IT',
    'OLMo-2-0325-32B': 'OLMo-2-32B-PT',
    'OLMo-2-0325-32B-Instruct': 'OLMo-2-32B-IT',
}


def display_name(short):
    return DISPLAY.get(short, short)


def dist_csv(m):
    return f'82.dist_{model_short(m)}.csv'

In [2]:
import importlib, figure_style; importlib.reload(figure_style)

<module 'figure_style' from 'code/figure_style.py'>

In [3]:
labels = sorted(f for f in os.listdir(TEXT_FOLDER) if os.path.isfile(os.path.join(TEXT_FOLDER, f)))
grouping = [f.split(',')[1] for f in labels]
pairs = list(combinations(labels, 2))
print(f"{len(labels)} texts, {len(pairs)} pairs, groups: {sorted(set(grouping))}")

51 texts, 1275 pairs, groups: ['mood', 'somatic', 'suicidality']


## Part 1: distance matrices

In [4]:
def energy(x, y):
    statistic, _, _ = pted(x, y, return_all=True, two_tailed=False, permutations=1)
    return statistic


def layer_distances(layer, clouds):
    """clouds: {text: (n_tokens, d) float64 tensor} at one layer -> list of long-format rows."""
    torch.set_num_threads(1)  # one layer per joblib worker; avoid oversubscribing the cores
    unit = {t: normalize(x) for t, x in clouds.items()}
    cent = {t: centroid(x) for t, x in clouds.items()}
    cent_unit = {t: centroid(x) for t, x in unit.items()}
    rows = []
    for f1, f2 in pairs:
        symptom = f1.split(',')[1] if f1.split(',')[1] == f2.split(',')[1] else None
        values = {
            'centroid_cosine': cos_dist(cent[f1], cent[f2]),
            'centroid_cosine_unit': cos_dist(cent_unit[f1], cent_unit[f2]),
            'centroid_euclidean': euc(cent[f1], cent[f2]),
            'centroid_euclidean_unit': euc(cent_unit[f1], cent_unit[f2]),
            'earth_mover': wasserstein_distance_nd(clouds[f1].numpy(), clouds[f2].numpy()),
            'earth_mover_unit': wasserstein_distance_nd(unit[f1].numpy(), unit[f2].numpy()),
            'energy': energy(clouds[f1], clouds[f2]),
            'energy_unit': energy(unit[f1], unit[f2]),
        }
        rows += [[dt, layer, f1, f2, symptom, float(values[dt])] for dt in DIST_TYPES]
    return rows


def load_stream(folder):
    """{text: [layer: (n_tokens - 1, d) float64]} with the BOS / attention sink dropped."""
    return {t: [r[1:].detach().to(torch.float64) for r in torch.load(os.path.join(folder, t + '.pt'), map_location='cpu')]
            for t in tqdm(labels, desc='load', leave=False)}

In [5]:
# Models with a complete full-stream folder (OLMo-2-0325-32B failed to generate in the full-stream run).
available = [m for m in MODELS
             if all(os.path.exists(os.path.join(output_folder_name(TEXT_FOLDER, m), t + '.pt')) for t in labels)]
print("available:", [model_short(m) for m in available])
print("missing:  ", [model_short(m) for m in MODELS if m not in available])

available: []
missing:   ['gemma-3-27b-pt', 'gemma-3-27b-it', 'gemma-3-12b-pt', 'gemma-3-12b-it', 'gemma-3-4b-pt', 'gemma-3-4b-it', 'Mistral-Small-24B-Base-2501', 'Mistral-Small-24B-Instruct-2501', 'OLMo-2-0325-32B', 'OLMo-2-0325-32B-Instruct']


In [6]:
# Distance CSVs that already exist are kept; delete one to recompute that model.
for m in available:
    if os.path.exists(dist_csv(m)):
        continue
    stream = load_stream(output_folder_name(TEXT_FOLDER, m))
    n_layers = len(stream[labels[0]])
    jobs = (delayed(layer_distances)(l, {t: stream[t][l] for t in labels}) for l in range(n_layers))
    rows = []
    for layer_rows in tqdm(Parallel(n_jobs=N_JOBS, return_as='generator')(jobs),
                           total=n_layers, desc=model_short(m)):
        rows += layer_rows
    dist = DataFrame(rows, columns=['dist_type', 'layer', 'text1', 'text2', 'symptom', 'distance'])
    dist.to_csv(dist_csv(m), index=True)
    del stream, rows, dist

## Part 2: PERMANOVA and PERMDISP per (model, metric, layer)

Same calls as 74: `permanova` and `permdisp(test='centroid')`, 9999 permutations, grouped
by symptom. Seeded so a rerun reproduces the p-values. Models already in
`82.permanova.csv` are skipped.

In [7]:
def square_matrix(original):
    square = (original
              .pivot(index='text1', columns='text2', values='distance')
              .reindex(index=labels, columns=labels))
    return square.combine_first(square.T).fillna(0.0).values.copy()


def separation(model, dt, layer, mat):
    dm = DistanceMatrix(mat, ids=labels)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        anova = permanova(dm, grouping, permutations=PERMU, seed=SEED)
        disp = permdisp(dm, grouping, test='centroid', permutations=PERMU, seed=SEED)
    return ([model_short(model), dt, layer, anova['test statistic'], anova['p-value']],
            [model_short(model), dt, layer, disp['test statistic'], disp['p-value']])


done = set(read_csv(PERMANOVA_CSV, index_col=0)['model']) if os.path.exists(PERMANOVA_CSV) else set()
for m in [m for m in available if model_short(m) not in done]:
    dist = read_csv(dist_csv(m), index_col=0)
    groups = list(dist.groupby(['dist_type', 'layer']))
    jobs = (delayed(separation)(m, dt, l, square_matrix(g)) for (dt, l), g in groups)
    results = list(tqdm(Parallel(n_jobs=N_JOBS, return_as='generator')(jobs),
                        total=len(groups), desc=model_short(m)))
    new_anova = DataFrame([r[0] for r in results], columns=['model', 'dist_type', 'layer', 'pseudo-F', 'p-value'])
    new_disp = DataFrame([r[1] for r in results], columns=['model', 'dist_type', 'layer', 'F-value', 'p-value'])
    # Append per model, so an interrupted run keeps the models already finished.
    for new, path in [(new_anova, PERMANOVA_CSV), (new_disp, PERMDISP_CSV)]:
        old = read_csv(path, index_col=0) if os.path.exists(path) else None
        pd.concat([old, new], ignore_index=True).to_csv(path, index=True)

permanova_pd = read_csv(PERMANOVA_CSV, index_col=0)
permdisp_pd = read_csv(PERMDISP_CSV, index_col=0)

## Part 3: gated peak per model and metric, and the operating layer

Eligible layers have PERMDISP p > 0.05; each metric's gated peak is the max pseudo-F among
them. `rel_depth` is layer / blocks, for comparing models of different depth.

**Operating layer** (one per model): the layer that the most metrics' gated peaks agree on.
Ties go to the layer where more metrics pass the PERMDISP gate. Part 4 generates the
single-layer residual streams there; the layers are also written to `82.operating_layers.csv`.

In [8]:
stats = (permanova_pd
         .merge(permdisp_pd.rename(columns={'F-value': 'permdisp_F', 'p-value': 'permdisp_p'}),
                on=['model', 'dist_type', 'layer'])
         .rename(columns={'pseudo-F': 'F', 'p-value': 'permanova_p'}))
stats['blocks'] = stats.groupby('model')['layer'].transform('max')
stats['rel_depth'] = stats['layer'] / stats['blocks']
stats['eligible'] = stats['permdisp_p'] > ALPHA

eligible = stats[stats['eligible']]
peaks = eligible.loc[eligible.groupby(['model', 'dist_type'])['F'].idxmax()]

model_order = [model_short(m) for m in MODELS if model_short(m) in set(stats['model'])]

summary = (peaks
           .assign(model=lambda d: pd.Categorical(d['model'], model_order),
                   dist_type=lambda d: pd.Categorical(d['dist_type'], DIST_TYPES))
           .sort_values(['dist_type', 'model'])
           .set_index(['dist_type', 'model'])
           [['layer', 'blocks', 'rel_depth', 'F', 'permanova_p', 'permdisp_F', 'permdisp_p']])
summary.to_csv(PEAKS_CSV)

# Operating layer per model (rule in the markdown above).
n_metrics = stats['dist_type'].nunique()
per_layer = (stats.groupby(['model', 'layer'])
             .agg(n_eligible=('eligible', 'sum'), blocks=('blocks', 'first'))
             .join(peaks.groupby(['model', 'layer']).size().rename('n_peaks'))
             .fillna({'n_peaks': 0})
             .reset_index())
ranked = per_layer.sort_values(['n_peaks', 'n_eligible'], ascending=False).groupby('model')
top2 = ranked.head(2).groupby('model')[['n_peaks', 'n_eligible']]
tied = [m for m, g in top2 if len(g) == 2 and g.iloc[0].equals(g.iloc[1])]
assert not tied, f"operating layer still tied after both criteria: {tied}"
operating = ranked.head(1).set_index('model').reindex(model_order)
operating['rel_depth'] = operating['layer'] / operating['blocks']
operating[['layer', 'blocks', 'rel_depth', 'n_peaks', 'n_eligible']].to_csv(OPERATING_CSV)

## Per-layer separation curves for all models

The gate-first per-layer figure from 74, with a dropdown to switch between models.
PERMDISP defines the eligible region at each layer (non-significant within-group dispersion).
The operating peak is the maximum PERMANOVA pseudo-F within that region; a metric's global
peak on an ineligible layer is shown but excluded. Panels are ordered by operating-peak pseudo-F.

In [9]:
import math
from plotly.subplots import make_subplots

FIG_SEPARATION_HTML = f'{TMLR_DIR}/82.multimodel_separation.html'

# Formal metric titles (same as 74)
FORMAL_TITLES = {
    'centroid_cosine': "Centroid Cosine<br>(raw)",
    'centroid_cosine_unit': "Centroid Cosine<br>(unit-normalized)",
    'centroid_euclidean': "Centroid Euclidean<br>(raw)",
    'centroid_euclidean_unit': "Centroid Euclidean<br>(unit-normalized)",
    'earth_mover': "Earth Mover's<br>(raw)",
    'earth_mover_unit': "Earth Mover's<br>(unit-normalized)",
    'energy': "Energy<br>(raw)",
    'energy_unit': "Energy<br>(unit-normalized)"
}

# Colours (same as 74)
COLOR_PRIMARY   = "#000000"                 # black — pseudo-F trajectory
COLOR_REF       = "rgba(150,150,150,0.4)"   # reference line at y=0
GATE_PASS  = GREEN                          # green — operating (eligible) peak
GATE_FAIL  = "#9a9a9a"                      # gray  — excluded (ineligible) peak
INADM_FILL = "rgba(213,94,0,0.25)"         # pale red — ineligible layer band


def _sig_runs(layers_sorted):
    """Contiguous runs of PERMDISP-significant layers -> [[start, end], ...]."""
    runs = []
    for L in layers_sorted:
        if runs and L == runs[-1][1] + 1:
            runs[-1][1] = L
        else:
            runs.append([L, L])
    return runs


def _annotation_offset(m_df, Lo, Fo, Y_MAX, window=8):
    """Return (ax, ay) pixel offsets that avoid overlapping the line."""
    left_f  = m_df.loc[(m_df['layer'] >= Lo - window) & (m_df['layer'] < Lo),  'F']
    right_f = m_df.loc[(m_df['layer'] >  Lo)          & (m_df['layer'] <= Lo + window), 'F']
    left_drop  = Fo - (left_f.mean()  if len(left_f)  else Fo)
    right_drop = Fo - (right_f.mean() if len(right_f) else Fo)
    ax = -52 if left_drop > right_drop else 52
    ay = 53 if Fo > Y_MAX * 7 / 8 else -10
    return ax, ay


# Precompute data for each model
model_data = {}
for mdl in model_order:
    perm = permanova_pd.query('model == @mdl').drop(columns='model')
    disp = permdisp_pd.query('model == @mdl').drop(columns='model')
    
    Y_MAX = math.ceil(perm['pseudo-F'].max() * 1.15)
    X_MAX = int(perm['layer'].max())
    
    metric_info = {}
    for dt in FORMAL_TITLES:
        if dt not in set(perm['dist_type']):
            continue
        pm = perm[perm['dist_type'] == dt][['layer', 'pseudo-F']].rename(columns={'pseudo-F': 'F'})
        pp = disp[disp['dist_type'] == dt][['layer', 'p-value']].rename(columns={'p-value': 'permdisp_p'})
        m = pm.merge(pp, on='layer').sort_values('layer').reset_index(drop=True)
        
        g = m.loc[m['F'].idxmax()]
        adm = m[m['permdisp_p'] > ALPHA]
        op = adm.loc[adm['F'].idxmax()] if len(adm) else None
        global_adm = bool(g['permdisp_p'] > ALPHA)
        runs = _sig_runs(sorted(m.loc[m['permdisp_p'] <= ALPHA, 'layer'].tolist()))
        
        metric_info[dt] = dict(m=m, g=g, op=op, global_adm=global_adm, runs=runs,
                               op_F=(float(op['F']) if op is not None else -1.0))
    
    ordered = sorted([dt for dt in FORMAL_TITLES if dt in metric_info], key=lambda d: -metric_info[d]['op_F'])
    model_data[mdl] = {'Y_MAX': Y_MAX, 'X_MAX': X_MAX, 'info': metric_info, 'ordered': ordered}

# Use the first model's metric order for consistent panel layout
base_ordered = model_data[model_order[0]]['ordered']
rows, cols = 3, 3

# Build figure with subplots
fig_separation = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[FORMAL_TITLES[dt] for dt in base_ordered],
    shared_xaxes=False, shared_yaxes=False,
    horizontal_spacing=0.05, vertical_spacing=0.15,
)
# Subplot-title slots. Each model orders its panels by its own operating pseudo-F, and a button's
# `annotations` replaces every layout annotation, so each model carries its own titles.
title_slots = [a.to_plotly_json() for a in fig_separation.layout.annotations]

# Add traces for all models
traces_per_model = []
all_annotations = {}

for mdl_idx, mdl in enumerate(model_order):
    data = model_data[mdl]
    Y_MAX, X_MAX = data['Y_MAX'], data['X_MAX']
    info = data['info']
    ordered = data['ordered']
    
    start_trace = len(fig_separation.data)
    vis = (mdl_idx == 0)
    annotations = [{**slot, 'text': FORMAL_TITLES[dt]} for slot, dt in zip(title_slots, ordered)]
    
    for i, dt in enumerate(ordered):
        row, col = i // cols + 1, i % cols + 1
        d = info[dt]
        m = d['m']
        
        # Ineligible bands
        for (a, b) in d['runs']:
            fig_separation.add_trace(go.Scatter(
                x=[a - 0.5, a - 0.5, b + 0.5, b + 0.5, a - 0.5],
                y=[0, Y_MAX, Y_MAX, 0, 0],
                fill='toself', fillcolor=INADM_FILL,
                line=dict(width=0), mode='lines',
                showlegend=False, hoverinfo='skip', visible=vis,
            ), row=row, col=col)
        
        # Pseudo-F trajectory
        fig_separation.add_trace(go.Scatter(
            x=m['layer'], y=m['F'], mode='lines',
            line=dict(color=COLOR_PRIMARY, width=1.4),
            hovertemplate='layer %{x}<br>pseudo-F %{y:.2f}<extra></extra>',
            showlegend=False, visible=vis), row=row, col=col)
        
        # Excluded global peak
        if not d['global_adm']:
            fig_separation.add_trace(go.Scatter(
                x=[d['g']['layer']], y=[d['g']['F']], mode='markers',
                marker=dict(symbol='circle-open', color=GATE_FAIL, size=8, line=dict(width=1.6)),
                showlegend=False, hoverinfo='skip', visible=vis), row=row, col=col)
        
        # Operating peak
        if d['op'] is not None:
            Lo, Fo = int(d['op']['layer']), float(d['op']['F'])
            fig_separation.add_trace(go.Scatter(
                x=[Lo], y=[Fo], mode='markers',
                marker=dict(color=GATE_PASS, size=7),
                showlegend=False, hoverinfo='skip', visible=vis), row=row, col=col)
            ax_val, ay_val = _annotation_offset(m, Lo, Fo, Y_MAX)
            xref = f"x{i+1}" if i > 0 else "x"
            yref = f"y{i+1}" if i > 0 else "y"
            annotations.append(dict(
                x=Lo, y=Fo, text=f"F = {Fo:.2f}<br>L{Lo}",
                showarrow=True, arrowhead=2, arrowwidth=0.6, arrowsize=1,
                arrowcolor=GATE_PASS, font={**TICK, "color": GATE_PASS},
                bgcolor='rgba(255,255,255,0.85)', bordercolor='rgba(0,0,0,0.12)',
                borderwidth=0.5, align='center', ax=ax_val, ay=ay_val,
                xref=xref, yref=yref))
    
    traces_per_model.append((start_trace, len(fig_separation.data)))
    all_annotations[mdl] = annotations

# Add legend proxy traces at the end (always visible)
fig_separation.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(symbol='square', color='rgba(213,94,0,0.20)', size=11),
    name='ineligible layer<br>(PERMDISP p ≤ 0.05)', visible=True))
fig_separation.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(color=GATE_PASS, size=9),
    name='operating peak', visible=True))
fig_separation.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(symbol='circle-open', color=GATE_FAIL, size=9, line=dict(width=1.6)),
    name='excluded global peak', visible=True))

n_legend = 3
total_traces = len(fig_separation.data)

# Build dropdown buttons
buttons = []
for k, mdl in enumerate(model_order):
    # Build visibility: model traces visible, others hidden, legend always visible
    vis_list = []
    for j, (start, end) in enumerate(traces_per_model):
        vis_list.extend([j == k] * (end - start))
    vis_list.extend([True] * n_legend)  # Legend proxies always visible
    
    # Build axis range updates
    axis_updates = {'annotations': all_annotations[mdl]}
    Y_MAX, X_MAX = model_data[mdl]['Y_MAX'], model_data[mdl]['X_MAX']
    for i in range(8):
        ax_key = f"xaxis{i+1}" if i > 0 else "xaxis"
        ay_key = f"yaxis{i+1}" if i > 0 else "yaxis"
        axis_updates[f'{ax_key}.range'] = [0, X_MAX]
        axis_updates[f'{ay_key}.range'] = [0, Y_MAX]
    
    buttons.append(dict(
        label=f"{display_name(mdl)} · L{int(operating.loc[mdl, 'layer'])}",
        method='update',
        args=[{'visible': vis_list}, axis_updates]
    ))

# Layout
fig_separation.update_layout(
    template='plotly_white',
    font=FONT,
    margin=dict(l=60, r=10, t=50, b=60),
    height=760,
    showlegend=True,
    # Legend in the empty ninth cell of the 3 x 3 grid: above the panels it collides with the
    # two-line panel titles once it wraps at column or phone width.
    legend=dict(orientation='v', x=fig_separation.layout.xaxis9.domain[0], xanchor='left',
                y=fig_separation.layout.yaxis9.domain[1], yanchor='top',
                font=TICK, bgcolor='rgba(0,0,0,0)', itemsizing='constant'),
    # Carries the per-model views; the export swaps it for an HTML <select> above the chart.
    updatemenus=[dict(buttons=buttons, direction='down', x=0, xanchor='left',
                      y=1.12, yanchor='bottom', font=TICK, showactive=True)],
    annotations=all_annotations[model_order[0]],
)

# Axis titles
bottom_row_of_col = {1: 3, 2: 3, 3: 2}  # col -> bottom row with content (8 panels in 3x3)
for c, r in bottom_row_of_col.items():
    fig_separation.update_xaxes(title_text="Transformer Layer", title_font=TITLE, row=r, col=c)
fig_separation.update_yaxes(title_text="PERMANOVA pseudo-F", title_font=TITLE, row=2, col=1)

# Axis styling with initial ranges from first model
Y_MAX_init, X_MAX_init = model_data[model_order[0]]['Y_MAX'], model_data[model_order[0]]['X_MAX']
fig_separation.update_xaxes(range=[0, X_MAX_init], tickfont=TICK, showgrid=False,
                            linecolor='black', linewidth=0.6, ticks='outside', ticklen=3, tickwidth=0.6)
fig_separation.update_yaxes(range=[0, Y_MAX_init], tickfont=TICK, showgrid=True,
                            gridcolor='rgba(200,200,200,0.3)', gridwidth=0.5,
                            linecolor='black', linewidth=0.6, ticks='outside', ticklen=3, tickwidth=0.6)

print(f"Built separation figure with {len(model_order)} models, {total_traces} traces")
fig_separation.show()

Built separation figure with 10 models, 435 traces


In [10]:
# Export the separation figure
EXPORT_SEPARATION = True
if EXPORT_SEPARATION:
    os.makedirs(os.path.dirname(FIG_SEPARATION_HTML), exist_ok=True)
    write_tmlr_dropdown_html(fig_separation, FIG_SEPARATION_HTML)
    print("wrote", FIG_SEPARATION_HTML)

wrote tmlr-beyond-pdf-author-kit/submission_folder/assets/html/submission/82.multimodel_separation.html


## Part 4: residual streams at each model's operating layer

For every corpus the projection and Depression Vector experiments use, the single-layer
residual stream at the operating layer from Part 3:

| Corpus | Used in |
|---|---|
| `core_clinical_short` | Symptom Vectors, depression pole |
| `books` | naturalistic, held-out |
| `redsm5` | naturalistic top-up |
| `positive_affect` | positive-affect pole |
| `happydb` | happy control, held-out |

Set `GENERATE_RS = True` on the Mac. Jobs whose output is complete are skipped (no model is
loaded for them), so an interrupted run can simply be re-run. Every file in each folder is
generated; subsampling of `redsm5` and `happydb` stays in the analysis. Output goes to
`<corpus>_<model>_layer<L>/` (`<t>_tensor`, `<t>_tokens`), the format 75 and 79 read. Stream
index L is the input to decoder block L, the same index Part 1 calls layer L.

In [11]:
from residual_stream import generate_missing, is_complete

RS_FOLDERS = ['core_clinical_short', 'books', 'redsm5', 'positive_affect', 'happydb']
GENERATE_RS = True  # loads every model in turn: run on the Mac, leave off for re-exports

# A model just added to MODELS has no operating layer until Parts 1-3 have run on it.
print("no operating layer yet:", [model_short(m) for m in MODELS if model_short(m) not in operating.index])
layer_jobs = [(m, folder, int(operating.loc[model_short(m), 'layer']))
              for m in MODELS if model_short(m) in operating.index
              for folder in RS_FOLDERS]
print(f"{sum(not is_complete(*job) for job in layer_jobs)} of {len(layer_jobs)} jobs left to run")

if GENERATE_RS:
    failed = generate_missing(layer_jobs, dtype=torch.bfloat16)  # bf16 for every model

no operating layer yet: []


0 of 50 jobs left to run

failed: {}


## Part 5: projection alignment and contrastive AUC at each operating layer

75's Gram-inverse projection and 79's contrastive depression direction, unchanged, on
the single-layer residual streams from Part 4. Per model:

- **Clinician alignment (of 3)**: on the held-out naturalistic corpus, rank the three
  annotation groups by median coefficient within each projection axis (75's Fig. 3 D); an
  annotation group aligns when it ranks 1st on its own axis (rank-1 diagonal cell).
- **Contrastive AUC**: held-out naturalistic vs HappyDB, Mann-Whitney AUC of the cosine to
  centroid(core clinical) - centroid(positive affect) (79's held-out comparison).

Sampling matches 75 / 79: books in full, redsm5 topped up to 141 per symptom (SEED 0),
HappyDB truncated to 423. An experiment is skipped for a model when any text it reads is
missing from that model's Part 4 folders.

In [12]:
SYMPTOMS = ['mood', 'somatic', 'suicidality']
SAMPLE_SEED = 0  # redsm5 top-up seed, as in 75 / 79

_books_n = defaultdict(int)
for _f in os.listdir('books'):
    _books_n[_f.split(',')[1]] += 1
_target = max(_books_n.values())  # 141 per symptom
_happy_n = 3 * _target            # 423


def select_files(text_dir):
    """Filenames to include for a corpus (deterministic given SAMPLE_SEED), as in 75 / 79."""
    files = sorted(os.listdir(text_dir))
    if text_dir == 'redsm5':
        by_symptom = defaultdict(list)
        for f in files:
            by_symptom[f.split(',')[1]].append(f)
        rng = random.Random(SAMPLE_SEED)
        selected = []
        for symptom in sorted(by_symptom):
            need = max(0, _target - _books_n[symptom])
            selected += rng.sample(by_symptom[symptom], min(need, len(by_symptom[symptom])))
        return sorted(selected)
    if text_dir == 'happydb':
        return files[:_happy_n]
    return files


def has_texts(m, layer, text_dirs):
    """True if every selected text of every corpus has its operating-layer tensor."""
    return all(os.path.exists(os.path.join(output_folder_name(d, m, layer), t + '_tensor'))
               for d in text_dirs for t in select_files(d))


def load_texts(m, layer, text_dir):
    """{text: (n_tokens - 1, d) float64} at the operating layer, BOS dropped, float64 BEFORE averaging."""
    folder = output_folder_name(text_dir, m, layer)
    return {t: torch.load(os.path.join(folder, t + '_tensor'), map_location='cpu')[0][1:].double()
            for t in select_files(text_dir)}


def symptom_centroids(core):
    """Token-weighted centroid per symptom over the core clinical texts."""
    return {s: centroid(torch.cat([x for t, x in core.items() if t.split(',')[1] == s])) for s in SYMPTOMS}


def projection(core, texts):
    """75: Gram-inverse coefficients on the core clinical Symptom Vectors, one row per text."""
    cent = symptom_centroids(core)
    symptom_stack = torch.stack([cent[s] for s in SYMPTOMS])
    gram_inv = torch.linalg.inv(symptom_stack @ symptom_stack.T)
    coef = DataFrame([(gram_inv @ (symptom_stack @ centroid(x))).tolist() for x in texts.values()],
                     columns=SYMPTOMS, index=list(texts))
    coef['clinician_annotation'] = [t.split(',')[1] for t in texts]
    return coef


def experiment_2(coef):
    """75: number of annotation groups whose median coefficient ranks 1st on their own axis."""
    rank = coef.groupby('clinician_annotation')[SYMPTOMS].median().rank(ascending=False, method='min')
    return int(sum(rank.loc[s, s] == 1 for s in SYMPTOMS))


def experiment_3(core, positive, naturalistic, happy):
    """79: held-out AUC of naturalistic > HappyDB on the cosine to the depression direction,
    and every text's score in the four corpora."""
    core_centroid = centroid(torch.stack(list(symptom_centroids(core).values())))
    positive_centroid = centroid(torch.cat(list(positive.values())))  # token-weighted
    unit_direction = core_centroid - positive_centroid
    unit_direction = unit_direction / unit_direction.norm()

    def scores(texts):
        return [(c @ unit_direction / c.norm()).item() for c in map(centroid, texts.values())]

    corpora = {'core_clinical': core, 'naturalistic': naturalistic, 'control_happy': happy,
               'positive_affect': positive}
    score = DataFrame([(t, d, v) for d, texts in corpora.items() for t, v in zip(texts, scores(texts))],
                      columns=['filename', 'dataset', 'depression'])
    by = score.groupby('dataset')['depression']
    x, y = by.get_group('naturalistic'), by.get_group('control_happy')
    return mannwhitneyu(x, y, alternative='greater').statistic / (len(x) * len(y)), score

In [13]:
EXP2_FOLDERS = ['core_clinical_short', 'books', 'redsm5']
EXP3_FOLDERS = EXP2_FOLDERS + ['positive_affect', 'happydb']

downstream = {}
for m in tqdm(MODELS, desc='models'):
    layer = int(operating.loc[model_short(m), 'layer'])
    if not has_texts(m, layer, EXP2_FOLDERS):
        continue
    core = load_texts(m, layer, 'core_clinical_short')
    naturalistic = {**load_texts(m, layer, 'books'), **load_texts(m, layer, 'redsm5')}
    coef = pd.concat([projection(core, core).assign(corpus='core_clinical'),
                      projection(core, naturalistic).assign(corpus='naturalistic')])
    cent = symptom_centroids(core)
    result = {'alignment': experiment_2(coef[coef['corpus'] == 'naturalistic']), 'coef': coef,
              'gram_cond': gram_condition_number(torch.stack([cent[s] for s in SYMPTOMS]))}
    if has_texts(m, layer, EXP3_FOLDERS):
        result['auc'], result['scores'] = experiment_3(core, load_texts(m, layer, 'positive_affect'),
                                                       naturalistic, load_texts(m, layer, 'happydb'))
    downstream[model_short(m)] = result
    del core, naturalistic

models:   0%|          | 0/10 [00:00<?, ?it/s]

In [14]:
# Within-source check: symptom group and source are associated in the
# Naturalistic corpus (ReDSM5 has no mood excerpts), so alignment is re-run within each source.
# Books: dv = Darkness Visible, hd = Handbook of Depression; ReDSM5 has somatic and suicidality only.
def source_of(t):
    return 'dv' if t.startswith('dv') else 'hd' if t.startswith('hd') else 'redsm5'


def own_axis_first(coef):
    """(groups ranking 1st on their own axis, groups present), over the groups present."""
    groups = [s for s in SYMPTOMS if (coef['clinician_annotation'] == s).any()]
    rank = coef.groupby('clinician_annotation')[groups].median().rank(ascending=False, method='min')
    return sum(int(rank.loc[s, s] == 1) for s in groups), len(groups)


def auc_greater(a, b):
    res = mannwhitneyu(a, b, alternative='greater')
    return res.statistic / (len(a) * len(b)), res.pvalue


source_rows = []
for mdl, res in downstream.items():
    nat = res['coef'][res['coef']['corpus'] == 'naturalistic'].copy()
    nat['source'] = [source_of(t) for t in nat.index]
    books, red = nat[nat['source'] != 'redsm5'], nat[nat['source'] == 'redsm5']
    row = {'model': display_name(mdl)}
    for name, sub in [('books', books), ('handbook', nat[nat['source'] == 'hd']),
                      ('darkness_visible', nat[nat['source'] == 'dv']), ('redsm5', red)]:
        row[f'{name} (n={len(sub)})'] = '%d / %d' % own_axis_first(sub)
    for ax, other in [('somatic', 'suicidality'), ('suicidality', 'somatic')]:
        auc, p = auc_greater(red.loc[red['clinician_annotation'] == ax, ax],
                             red.loc[red['clinician_annotation'] == other, ax])
        row[f'redsm5 {ax}-axis AUC'], row[f'redsm5 {ax}-axis p'] = round(auc, 3), p
    for name, sub in [('books', books), ('redsm5', red)]:
        row[f'suicidality median, {name}'] = round(sub.loc[sub['clinician_annotation'] == 'suicidality', 'suicidality'].median(), 3)
    source_rows.append(row)
source_check = DataFrame(source_rows).set_index('model')
source_check


,books (n=229),handbook (n=172),darkness_visible (n=57),redsm5 (n=194),redsm5 somatic-axis AUC,redsm5 somatic-axis p,redsm5 suicidality-axis AUC,redsm5 suicidality-axis p,"suicidality median, books","suicidality median, redsm5"
model,,,,,,,,,,
Gemma-3-27B-PT,3 / 3,3 / 3,1 / 3,2 / 2,0.945,7.783268e-27,0.737,7.249323e-09,1.004,1.503
Gemma-3-27B-IT,3 / 3,3 / 3,3 / 3,2 / 2,0.809,6.883384e-14,0.733,1.167149e-08,0.865,1.564
Gemma-3-12B-PT,3 / 3,3 / 3,3 / 3,2 / 2,0.942,1.592430e-26,0.988,7.569483e-32,0.528,0.950
Gemma-3-12B-IT,3 / 3,3 / 3,3 / 3,2 / 2,0.957,3.585659e-28,0.994,1.465389e-32,0.498,0.898
Gemma-3-4B-PT,3 / 3,3 / 3,3 / 3,2 / 2,0.903,2.198864e-22,0.992,2.179649e-32,0.491,0.631
Gemma-3-4B-IT,3 / 3,3 / 3,3 / 3,2 / 2,0.933,1.560907e-25,0.994,1.219568e-32,0.455,0.638
Mistral-24B-PT,3 / 3,3 / 3,3 / 3,2 / 2,0.930,3.395971e-25,0.988,7.125290e-32,0.425,0.715
Mistral-24B-IT,3 / 3,3 / 3,3 / 3,2 / 2,0.925,1.280567e-24,0.989,5.764878e-32,0.439,0.729
OLMo-2-32B-PT,3 / 3,3 / 3,3 / 3,2 / 2,0.926,9.082619e-25,0.986,1.190367e-31,0.427,0.694


In [15]:
# Released vectors: each model's Symptom Vectors, their Gram inverse
# and the Depression Vector at its operating layer, float64, so new text can be scored without
# re-running Parts 1-4. NumPy .npz rather than .pt, which is gitignored. Usage: symptom_vectors/README.md.
import numpy as np

VECTORS_DIR = 'symptom_vectors'
os.makedirs(VECTORS_DIR, exist_ok=True)
for m in MODELS:
    mdl = model_short(m)
    if 'auc' not in downstream.get(mdl, {}):
        continue
    layer = int(operating.loc[mdl, 'layer'])
    cent = symptom_centroids(load_texts(m, layer, 'core_clinical_short'))
    positive = centroid(torch.cat(list(load_texts(m, layer, 'positive_affect').values())))  # token-weighted
    symptom_vectors = torch.stack([cent[s] for s in SYMPTOMS])
    np.savez(os.path.join(VECTORS_DIR, f'{mdl}_layer{layer}.npz'),
             model=m, layer=layer, symptoms=SYMPTOMS, symptom_vectors=symptom_vectors.numpy(),
             gram_inv=torch.linalg.inv(symptom_vectors @ symptom_vectors.T).numpy(),
             depression_vector=(symptom_vectors.mean(0) - positive).numpy())
print(sorted(os.listdir(VECTORS_DIR)))

['Mistral-Small-24B-Base-2501_layer8.npz', 'Mistral-Small-24B-Instruct-2501_layer8.npz', 'OLMo-2-0325-32B-Instruct_layer16.npz', 'OLMo-2-0325-32B_layer16.npz', 'README.md', 'gemma-3-12b-it_layer14.npz', 'gemma-3-12b-pt_layer13.npz', 'gemma-3-27b-it_layer21.npz', 'gemma-3-27b-pt_layer21.npz', 'gemma-3-4b-it_layer20.npz', 'gemma-3-4b-pt_layer20.npz']


## Per-model figures and tables

Every per-model figure and table,
built for each model at its operating layer from the Part 3 statistics and the Part 5 coefficients,
each with a model dropdown. Exported by the last cell.

In [16]:
# Shared by the per-model figures below: output paths, corpus colours and labels, tooltip text.
import textwrap

from plotly.subplots import make_subplots

from figure_style import GREEN, TITLE

FIG_MODELS_HTML = f'{TMLR_DIR}/82.multimodel_projection.html'
FIG_MODELS_3D_HTML = f'{TMLR_DIR}/82.multimodel_3d.html'
FIG_MODELS_AUC_HTML = f'{TMLR_DIR}/82.multimodel_auc.html'
WITHHELD_NOTE = '(text withheld: not redistributable)'  # books and redsm5, as in 75 / 79

# (a) subspace, as in 75: colour = clinician annotation, marker = corpus
SYMPTOM_COLOR = {'mood': '#0072B2', 'somatic': GREEN, 'suicidality': '#D55E00'}  # Okabe-Ito
SUBSPACE_CORPUS = {'core_clinical': ('Core Clinical · In-Sample', 'square', 'core_clinical_short'),
                   'naturalistic': ('Naturalistic · Held-Out', 'circle', None)}
# (b) Depression Score, as in 79: corpus -> (x label, colour, text folder; None = withheld)
DEPRESSIVE, POSITIVE = '#0072B2', '#CC79A7'
# Ordered by split, each contrast's pair adjacent, so the two brackets sit side by side, not nested.
SCORE_CORPUS = {'core_clinical': ('Core Clinical<br>(Depressed)', DEPRESSIVE, 'core_clinical_short'),
                'positive_affect': ('Positive<br>Affect', POSITIVE, 'positive_affect'),
                'naturalistic': ('Naturalistic<br>(Depressed)', DEPRESSIVE, None),
                'control_happy': ('HappyDB', POSITIVE, 'happydb')}
SCORE_ORDER = list(SCORE_CORPUS)
# (a, b), bracket height above the data (fraction of the span), label, split: as in 79
BRACKETS = [(('core_clinical', 'positive_affect'), 0.10, 'Core Clinical > Positive Affect', 'In-Sample'),
            (('naturalistic', 'control_happy'), 0.10, 'Naturalistic > HappyDB', 'Held-Out')]


def stars(p):
    """Significance stars, as in 79."""
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'


def passage(t, text_dir):
    """Wrapped passage for the tooltip, or the withheld note."""
    if text_dir is None:
        return WITHHELD_NOTE
    with open(os.path.join(text_dir, t), encoding='utf-8') as f:
        return '<br>'.join(textwrap.wrap(f.read(), width=70))


models = [mdl for mdl in model_order if 'scores' in downstream.get(mdl, {})]


In [17]:
import numpy as np
import plotly.express as px
from scipy.stats import norm, rankdata

from figure_style import GREEN_TINTS

TABLE_PERMANOVA_HTML = f'{TMLR_DIR}/82.multimodel_table_permanova.html'
FIG_COEF_ABC_HTML = f'{TMLR_DIR}/82.multimodel_coefficient_abc.html'
FIG_COEF_RANK_HTML = f'{TMLR_DIR}/82.multimodel_coefficient_rank.html'

MODEL_LABEL = {mdl: f"{display_name(mdl)} · L{int(operating.loc[mdl, 'layer'])}" for mdl in model_order}
coef_models = [mdl for mdl in model_order if mdl in downstream]
ANNOTATION_LABEL = {s: s.capitalize() for s in SYMPTOMS}
DATASETS = ['core_clinical', 'naturalistic']


def model_dropdown(figs, models):
    """One figure from per-model figures on the same subplot grid: model k's traces, shapes,
    annotations and axis ranges / category orders, one updatemenu button per model. The export
    (write_tmlr_dropdown_html) turns the buttons into an HTML <select> above the chart, so no
    room is reserved for the menu here."""
    merged = go.Figure(layout=figs[0].layout)
    visible, views = [], []
    for f in figs:
        merged.add_traces(list(f.data))
        visible.append([True if t.visible is None else t.visible for t in f.data])
        lay = f.layout.to_plotly_json()
        view = {'shapes': lay.get('shapes', []), 'annotations': lay.get('annotations', [])}
        for key, axis in lay.items():
            if key.startswith(('xaxis', 'yaxis')):
                view.update({f'{key}.{prop}': axis[prop] for prop in ('range', 'categoryarray') if prop in axis})
        views.append(view)

    def show(k):
        return [v if j == k else False for j, vis in enumerate(visible) for v in vis]

    for t, v in zip(merged.data, show(0)):
        t.visible = v
    merged.update_layout(updatemenus=[dict(
        buttons=[dict(label=MODEL_LABEL[mdl], method='update', args=[{'visible': show(k)}, views[k]])
                 for k, mdl in enumerate(models)],
        direction='down', x=0, xanchor='left', y=1, yanchor='bottom', font=TICK, showactive=True)])
    return merged


def projection_frame(mdl):
    """75's projection_gram for one model: one row per passage, with display labels and hover text."""
    pg = downstream[mdl]['coef'].rename_axis('filename').reset_index().rename(columns={'corpus': 'dataset'})
    pg['Annotation'] = pg['clinician_annotation'].map(ANNOTATION_LABEL)
    pg['hover_text'] = [passage(t, SUBSPACE_CORPUS[d][2]) for t, d in zip(pg['filename'], pg['dataset'])]
    return pg


projection_frames = {mdl: projection_frame(mdl) for mdl in coef_models}

In [18]:
# Per model (74's Table 1): PERMANOVA and PERMDISP at each metric's operating peak,
# metrics in 74's fixed order. p-values have a floor of
# 1 / (PERMU + 1) from the permutation count.
P_FLOOR = 1.0 / (PERMU + 1)


def fmt_p(p):
    return f"< {P_FLOOR:.4f}" if p <= P_FLOOR + 1e-12 else f"= {p:.4f}"


def permanova_table(mdl):
    rows = []
    for dt in DIST_TYPES:
        op = model_data[mdl]['info'][dt]['op']
        if op is None:
            continue
        r = stats[(stats['model'] == mdl) & (stats['dist_type'] == dt) & (stats['layer'] == op['layer'])].iloc[0]
        metric, norm_ = FORMAL_TITLES[dt].split('<br>')
        rows.append([f"{metric}<br><span style='color:#555'>{norm_.strip('()')}</span>", int(op['layer']),
                     f"{r['F']:.2f}<br><span style='color:#555'>p {fmt_p(r['permanova_p'])}</span>",
                     f"{r['permdisp_F']:.2f}<br><span style='color:#555'>p {fmt_p(r['permdisp_p'])}</span>"])
    fig = go.Figure(go.Table(
        columnwidth=[1.7, 0.6, 1.1, 1.1],
        header=dict(values=["<b>Metric</b>", "<b>Layer</b>", "<b>PERMANOVA</b><br>pseudo-F", "<b>PERMDISP</b><br>F"],
                    fill_color="#f3f3f3", line_color="#cccccc", align=["left", "center", "center", "center"],
                    font=TICK, height=44),
        cells=dict(values=list(zip(*rows)), fill_color="white", line_color="#e5e5e5",
                   align=["left", "center", "center", "center"], font=TICK, height=44),
    ))
    # Height for all eight metrics, so the frame does not change between models.
    fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), height=44 + 48 * len(DIST_TYPES) + 30)
    return fig


fig_table_permanova = model_dropdown([permanova_table(mdl) for mdl in model_order], model_order)
fig_table_permanova.show()

In [19]:
# Per model (75's Fig. 3 a-c): coefficient box plots in a 2 x 3 grid, symptom
# axes across, corpora down, y shared down each column, Dunn's pairwise brackets (BH-corrected over
# each corpus's 3 pairs x 3 axes) on every facet. Geometry and rationale as in 75.
COLOR_MAP = {ANNOTATION_LABEL[s]: c for s, c in SYMPTOM_COLOR.items()}
FIG_H = 560  # height only: the width follows the container
FACET_COL_SPACING, FACET_ROW_SPACING = 0.115, 0.10
BAND_FILL = "rgba(0,0,0,0.035)"
BAND_PAD_L, BAND_PAD_R = 0.030, 0.010
BRACKET_BASE_PX, BRACKET_GAP_PX, TICK_DROP_PX, LABEL_HEAD_PX = 12, 19, 5, 13
N_ROWS, N_COLS = len(DATASETS), len(SYMPTOMS)
PAIRS = [(0, 1), (1, 2), (0, 2)]  # annotation-group x positions, stacked low -> high


def dunn_p(groups):
    """Tie-corrected Dunn z-test: two-sided p for each pair (i<j)."""
    allv = np.concatenate(groups); N = allv.size
    ranks = rankdata(allv)
    edges = np.cumsum([0] + [g.size for g in groups])
    rbar = [ranks[edges[k]:edges[k + 1]].mean() for k in range(len(groups))]
    n = [g.size for g in groups]
    _, cnt = np.unique(allv, return_counts=True)
    sigma2 = (N * (N + 1) / 12.0) - np.sum(cnt ** 3 - cnt) / (12.0 * (N - 1))
    return {(i, j): 2 * norm.sf(abs((rbar[i] - rbar[j]) / np.sqrt(sigma2 * (1 / n[i] + 1 / n[j]))))
            for i in range(len(groups)) for j in range(i + 1, len(groups))}


def bh(pvals):
    """Benjamini-Hochberg FDR-adjusted p-values."""
    p = np.asarray(pvals, float); m = p.size; o = np.argsort(p); r = p[o]
    adj = np.minimum.accumulate((r * m / np.arange(1, m + 1))[::-1])[::-1]
    out = np.empty(m); out[o] = np.clip(adj, 0, 1)
    return out


def dataset_p_adj(pg, dataset):
    """{(axis, pair): BH-adjusted Dunn p} over one corpus's 9 tests."""
    sub = pg[pg['dataset'] == dataset]
    raw = {}
    for ax in SYMPTOMS:
        p = dunn_p([sub.loc[sub['clinician_annotation'] == an, ax].to_numpy() for an in SYMPTOMS])
        raw.update({(ax, pr): p[pr] for pr in PAIRS})
    return dict(zip(raw, bh(list(raw.values()))))


def coefficient_grid(pg):
    long = pg.melt(id_vars=['filename', 'clinician_annotation', 'Annotation', 'dataset', 'hover_text'],
                   value_vars=SYMPTOMS, var_name='symptom_axis', value_name='coefficient')
    n = pg.groupby(['dataset', 'clinician_annotation']).size()

    def annot_label(d, a):
        # Group name as an invisible prefix keeps the category unique per group; only n= shows.
        return f"<span style='font-size:0px'>{ANNOTATION_LABEL[a]}</span>n={n[(d, a)]}"

    long['annot_label'] = [annot_label(d, a) for d, a in zip(long['dataset'], long['clinician_annotation'])]
    col_labels = {d: [annot_label(d, a) for a in SYMPTOMS if (d, a) in n] for d in DATASETS}

    fig = px.box(long, y='coefficient', x='annot_label', color='Annotation',
                 facet_col='symptom_axis', facet_row='dataset',
                 facet_col_spacing=FACET_COL_SPACING, facet_row_spacing=FACET_ROW_SPACING,
                 points='all', hover_data=['filename', 'hover_text'],
                 category_orders={'symptom_axis': SYMPTOMS, 'dataset': DATASETS,
                                  'Annotation': list(COLOR_MAP)},
                 color_discrete_map=COLOR_MAP)

    # Facet axes are numbered row-major from the BOTTOM row: r = 0 is naturalistic.
    def ax_key(r, c, kind):
        idx = r * N_COLS + c + 1
        return f"{kind}axis" if idx == 1 else f"{kind}axis{idx}"

    def row_of(dataset):
        return N_ROWS - 1 - DATASETS.index(dataset)

    fig.update_xaxes(matches=None)
    fig.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels=True))
    for d in DATASETS:
        for c in range(N_COLS):
            fig.layout[ax_key(row_of(d), c, 'x')].update(categoryorder='array', categoryarray=col_labels[d])
    fig.update_yaxes(matches=None)
    fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))

    fig.update_layout(
        template='plotly_white', font=FONT, showlegend=True,
        legend=dict(orientation='h', x=0.5, xanchor='center', y=1.075, yanchor='bottom',
                    title=dict(text='Clinician annotation  ', font=TICK), font=TICK,
                    bgcolor='rgba(0,0,0,0)', itemclick='toggleothers'),
        boxmode='overlay', boxgap=0.4, boxgroupgap=0.3,
        margin=dict(l=70, r=40, t=36, b=60), height=FIG_H,
    )
    fig.update_traces(offsetgroup=None, alignmentgroup=None,
                      marker=dict(size=2.5, opacity=0.5, line=dict(width=0)),
                      line=dict(width=1.4), jitter=0.3, pointpos=0)

    letter = dict(zip(SYMPTOMS, ['(a)', '(b)', '(c)']))
    corpus_label = {d: SUBSPACE_CORPUS[d][0] for d in DATASETS}

    def tidy_facet(a):
        label = a.text.split('=')[-1]
        if label in SYMPTOMS:
            a.update(text=f"<b>{letter[label]}</b>  {ANNOTATION_LABEL[label]} Axis", font=TITLE)
        else:
            a.update(text=corpus_label.get(label, label), font=FONT)

    fig.for_each_annotation(tidy_facet)
    fig.update_xaxes(title_text='', tickfont={**TICK, 'size': TICK['size'] - 2}, tickangle=0, showgrid=False,
                     linecolor='black', linewidth=0.6, ticks='outside', ticklen=3, tickwidth=0.6, automargin=True)
    fig.update_yaxes(title_text='', title_font=TITLE, tickfont=TICK, title_standoff=4, automargin=True,
                     showgrid=True, gridcolor='rgba(200,200,200,0.3)', gridwidth=0.5, zeroline=True,
                     zerolinecolor='rgba(150,150,150,0.4)', zerolinewidth=0.5,
                     linecolor='black', linewidth=0.6, ticks='outside', ticklen=3, tickwidth=0.6)
    for r in range(N_ROWS):
        fig.layout[ax_key(r, 0, 'y')].update(title_text='Coefficient (β)', title_font=TITLE)

    # Brackets solved in pixels (75): R = 1.05*span / (1 - (B + 2G + L)/H), one pixel = R/H data units.
    p_adj = {d: dataset_p_adj(pg, d) for d in DATASETS}
    facet_h_px = (FIG_H - 36 - 60) * (1 - FACET_ROW_SPACING) / N_ROWS
    for c, axis in enumerate(SYMPTOMS):
        lo, hi = float(pg[axis].min()), float(pg[axis].max())
        span = hi - lo
        reserved = (BRACKET_BASE_PX + 2 * BRACKET_GAP_PX + LABEL_HEAD_PX) / facet_h_px
        unit = (1.05 * span / (1 - reserved)) / facet_h_px
        base, step, drop = BRACKET_BASE_PX * unit, BRACKET_GAP_PX * unit, TICK_DROP_PX * unit
        needed_hi = hi
        for d in DATASETS:
            idx = row_of(d) * N_COLS + c + 1
            xref, yref = f"x{idx}", f"y{idx}"
            top = float(pg.loc[pg['dataset'] == d, axis].max())
            for lvl, (x0, x1) in enumerate(PAIRS):
                y = top + base + lvl * step
                fig.add_shape(type='line', xref=xref, yref=yref, x0=x0, x1=x1, y0=y, y1=y,
                              line=dict(color='black', width=0.9))
                for xx in (x0, x1):
                    fig.add_shape(type='line', xref=xref, yref=yref, x0=xx, x1=xx, y0=y, y1=y - drop,
                                  line=dict(color='black', width=0.9))
                txt = stars(p_adj[d][(axis, (x0, x1))])
                fig.add_annotation(xref=xref, yref=yref, x=(x0 + x1) / 2, y=y, yshift=(2 if txt == 'ns' else -3),
                                   text=txt, showarrow=False, xanchor='center', yanchor='bottom',
                                   font={**TICK, 'size': TICK['size'] if txt != 'ns' else TICK['size'] - 2})
            needed_hi = max(needed_hi, top + base + 2 * step + LABEL_HEAD_PX * unit)
        for r in range(N_ROWS):
            fig.layout[ax_key(r, c, 'y')].range = [lo - 0.05 * span, needed_hi]

    for c in range(N_COLS):  # one faint band per column, spanning both rows and its title
        x0, x1 = fig.layout[ax_key(0, c, 'x')].domain
        fig.add_shape(type='rect', xref='paper', yref='paper', layer='below',
                      x0=x0 - BAND_PAD_L, x1=x1 + BAND_PAD_R, y0=-0.10, y1=1.045,
                      fillcolor=BAND_FILL, line=dict(width=0))
    return fig


fig_coef_abc = model_dropdown([coefficient_grid(projection_frames[mdl]) for mdl in coef_models], coef_models)
fig_coef_abc.show()

In [20]:
# Per model (75's Fig. 3 d): within each projection axis the three annotation
# groups are ranked by median β (rank 1 = highest); the outlined diagonal is each group on its
# own axis. Core clinical is the in-sample check, naturalistic the held-out test.
PANELS = [('core_clinical', 'Core Clinical<br>In-sample'), ('naturalistic', 'Naturalistic<br>Held-out')]
RANK_FILL = {1: GREEN_TINTS[2], 2: GREEN_TINTS[1], 3: GREEN_TINTS[0]}
# Colour scale runs over 4 - rank, so rank 1 is the darkest.
RANK_COLORSCALE = [[0, RANK_FILL[3]], [1 / 3, RANK_FILL[3]], [1 / 3, RANK_FILL[2]],
                   [2 / 3, RANK_FILL[2]], [2 / 3, RANK_FILL[1]], [1, RANK_FILL[1]]]
ORDINAL = {1: '1st', 2: '2nd', 3: '3rd'}


def rank_panel(pg, dataset):
    """z = 4 - rank, text = median β / #rank, hover = median, IQR, n, rank for each cell."""
    sub = pg[pg['dataset'] == dataset]
    rank = sub.groupby('clinician_annotation')[SYMPTOMS].median().rank(ascending=False, method='min')
    z, text, hover = [], [], []
    for an in SYMPTOMS:
        grp = sub[sub['clinician_annotation'] == an]
        z.append([]), text.append([]), hover.append([])
        for ax_ in SYMPTOMS:
            rk = int(rank.loc[an, ax_])
            q1, med, q3 = grp[ax_].quantile([0.25, 0.5, 0.75])
            z[-1].append(4 - rk)
            text[-1].append(f"{med:+.2f}<br><span style='font-size:10px'>#{rk}</span>")
            hover[-1].append(f"{ANNOTATION_LABEL[an]} texts (n = {len(grp)})<br>β {ANNOTATION_LABEL[ax_]}: "
                             f"median {med:+.2f} (Q1 {q1:+.2f}, Q3 {q3:+.2f})<br>{ORDINAL[rk]} of 3 on this axis")
    return z, text, hover


def rank_figure(pg):
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=[t for _, t in PANELS])
    for c, (dataset, _) in enumerate(PANELS, start=1):
        z, text, hover = rank_panel(pg, dataset)
        fig.add_trace(go.Heatmap(
            z=z, x=[ANNOTATION_LABEL[a] for a in SYMPTOMS], y=[ANNOTATION_LABEL[a] for a in SYMPTOMS],
            zmin=0.5, zmax=3.5, colorscale=RANK_COLORSCALE, xgap=1, ygap=1,
            text=text, texttemplate='%{text}', textfont=FONT, customdata=hover,
            hovertemplate='%{customdata}<extra></extra>', showscale=False,
        ), row=1, col=c)
        for i in range(len(SYMPTOMS)):  # outline the matching annotation / axis cells
            fig.add_shape(type='rect', x0=i - 0.5, x1=i + 0.5, y0=i - 0.5, y1=i + 0.5,
                          xref=f"x{'' if c == 1 else c}", yref=f"y{'' if c == 1 else c}",
                          line=dict(color='#1a1a1a', width=2))
    fig.update_yaxes(autorange='reversed', showgrid=False)
    fig.update_yaxes(title_text='Clinician annotation', row=1, col=1)
    fig.update_yaxes(showticklabels=False, row=1, col=2)
    fig.update_xaxes(side='top', showgrid=False, tickangle=0, tickfont=TICK,
                     title_text='Projection axis (β)', title_font=TICK, title_standoff=2)
    fig.update_layout(template='plotly_white', font=FONT, height=336, margin=dict(l=10, r=10, t=80, b=10))
    fig.update_annotations(yshift=36, font=TITLE)  # two-line panel titles above the axis headers
    return fig


fig_coef_rank = model_dropdown([rank_figure(projection_frames[mdl]) for mdl in coef_models], coef_models)
fig_coef_rank.show()

In [21]:
# Per model (75's 3D figure): passages in the three-dimensional symptom subspace.
fig_3d = go.Figure()
n_traces_3d = 2 * len(SYMPTOMS)  # 6 traces per model (2 corpora x 3 symptoms)

for k, mdl in enumerate(models):
    coef = downstream[mdl]['coef']
    for corpus, (corpus_label, symbol, text_dir) in SUBSPACE_CORPUS.items():
        for s in SYMPTOMS:
            g = coef[(coef['corpus'] == corpus) & (coef['clinician_annotation'] == s)]
            vis = k == 0  # the first model shows; the dropdown selects the others
            fig_3d.add_trace(go.Scatter3d(
                x=g['mood'], y=g['somatic'], z=g['suicidality'], mode='markers', visible=vis,
                name=f"{s.capitalize()}, {corpus_label}",
                marker=dict(size=4, color=SYMPTOM_COLOR[s], symbol=symbol, opacity=0.85, line=dict(width=0)),
                customdata=[[t, passage(t, text_dir)] for t in g.index],
                hovertemplate='%{customdata[0]}<br>β mood %{x:.2f} · somatic %{y:.2f} · '
                              'suicidality %{z:.2f}<br>%{customdata[1]}<extra></extra>'))

# Visibility lists for the dropdown: both corpora of the selected model show
def make_vis_3d(model_idx):
    vis = []
    for k in range(len(models)):
        for corpus in SUBSPACE_CORPUS:
            for _ in SYMPTOMS:
                vis.append(k == model_idx)
    return vis

buttons_3d = [dict(label=f"{display_name(mdl)} · L{int(operating.loc[mdl, 'layer'])}", method='update',
                   args=[{'visible': make_vis_3d(k)}])
              for k, mdl in enumerate(models)]

SCENE_AXIS = dict(backgroundcolor='rgba(0,0,0,0)', gridcolor='rgba(200,200,200,0.3)',
                  zerolinecolor='rgba(150,150,150,0.4)', linecolor='black', linewidth=0.75,
                  title_font=TITLE, tickfont=TICK)
fig_3d.update_layout(
    template='plotly_white', font=FONT, height=500,
    margin=dict(l=10, r=10, t=10, b=10),
    updatemenus=[dict(buttons=buttons_3d, direction='down', x=0, xanchor='left', y=1.02, yanchor='bottom',
                      font=TICK, showactive=True)],
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.02, yanchor='top',
                font=TICK, itemsizing='constant'),
    # Close enough to fill the frame, far enough that every point and all three axis titles show
    scene=dict(camera=dict(eye=dict(x=1.66, y=1.66, z=0.85), center=dict(x=0, y=0, z=-0.08)),
               aspectmode='cube',
               xaxis=dict(title_text='Mood', **SCENE_AXIS),
               yaxis=dict(title_text='Somatic', **SCENE_AXIS),
               zaxis=dict(title_text='Suicidality', **SCENE_AXIS)),
)

fig_3d.show()

In [22]:
# Depression Score (79's figure) by corpus, one panel per model at its operating layer,
# AUC_COLS panels per row. y is per panel: the score scale differs between model families.
AUC_COLS = 3
AUC_PANEL_PX, AUC_GAP_PX = 200, 80  # the gap fits two-line tick labels and the next row's titles
SCORE_TICK = {'core_clinical': 'Core<br>Clinical', 'positive_affect': 'Positive<br>Affect',
              'naturalistic': 'Natur-<br>alistic', 'control_happy': 'HappyDB'}
SMALL = {**TICK, 'size': TICK['size'] - 1}

auc_rows = math.ceil(len(models) / AUC_COLS)
auc_plot_px = AUC_PANEL_PX * auc_rows + AUC_GAP_PX * (auc_rows - 1)
fig_auc = make_subplots(rows=auc_rows, cols=AUC_COLS, horizontal_spacing=0.06, vertical_spacing=AUC_GAP_PX / auc_plot_px,
                        subplot_titles=[f"{display_name(mdl)} · L{int(operating.loc[mdl, 'layer'])}" for mdl in models])

for k, mdl in enumerate(models):
    row, col = k // AUC_COLS + 1, k % AUC_COLS + 1
    xref, yref = f"x{k + 1 if k else ''}", f"y{k + 1 if k else ''}"
    by = {d: g for d, g in downstream[mdl]['scores'].groupby('dataset')}
    for d in SCORE_ORDER:
        label, color, text_dir = SCORE_CORPUS[d]
        g = by[d]
        fig_auc.add_trace(go.Box(
            x=[SCORE_TICK[d]] * len(g), y=g['depression'], name=label.replace('<br>', ' '), showlegend=False,
            boxpoints='all', jitter=0.35, pointpos=0, line=dict(width=1.4, color=color),
            marker=dict(size=2.5, opacity=0.45, color=color, line=dict(width=0)),
            customdata=[[t, passage(t, text_dir)] for t in g['filename']],
            hovertemplate=f"{label.replace('<br>', ' ')} (n={len(g)})<br>"
                          '%{customdata[0]}<br>cosine %{y:.4f}<br>%{customdata[1]}<extra></extra>'),
            row=row, col=col)

    values = downstream[mdl]['scores']['depression']
    lo, hi = values.min(), values.max()
    span = hi - lo
    for (a, b), lift, kind, split in BRACKETS:
        res = mannwhitneyu(by[a]['depression'], by[b]['depression'], alternative='greater')
        u = res.statistic / (len(by[a]) * len(by[b]))
        y, i, j = hi + span * lift, SCORE_ORDER.index(a), SCORE_ORDER.index(b)
        fig_auc.add_shape(type='path', xref=xref, yref=yref, line=dict(color='black', width=0.9),
                          path=f"M {i},{y - span * 0.025} L {i},{y} L {j},{y} L {j},{y - span * 0.025}")
        fig_auc.add_annotation(xref=xref, yref=yref, x=(i + j) / 2, y=y, yanchor='bottom', showarrow=False,
                               font=SMALL, align='center', text=f"{split}<br>AUC {u:.3f} {stars(res.pvalue)}")
    fig_auc.update_yaxes(range=[lo - span * 0.06, hi + span * 0.42], row=row, col=col)

fig_auc.update_xaxes(tickfont=SMALL, tickangle=0, showgrid=False, linecolor='black', linewidth=0.8,
                     ticks='outside', ticklen=3, tickwidth=0.8)
fig_auc.update_yaxes(tickfont=SMALL, showgrid=True, gridcolor='rgba(200,200,200,0.3)', gridwidth=0.6,
                     zeroline=False, linecolor='black', linewidth=0.8, ticks='outside', ticklen=3, tickwidth=0.8)
for r in range(1, auc_rows + 1):
    fig_auc.update_yaxes(title_text='Cosine similarity to<br>depression direction', title_font=TICK,
                         title_standoff=4, row=r, col=1)
fig_auc.update_annotations(selector=dict(xref='paper'), font=TICK)  # subplot titles
fig_auc.update_layout(template='plotly_white', font=FONT, height=auc_plot_px + 60,
                      margin=dict(l=60, r=10, t=40, b=20), boxmode='overlay', boxgap=0.4)

fig_auc.show()

## Model comparison

Experiment 1 (Parts 1-3): operating layer, its relative depth, how many of the 8 metrics'
gated peaks sit there, and the highest pseudo-F at that layer among the metrics passing the
PERMDISP gate there. Experiment 2: annotation groups ranking 1st on their own projection axis
(held-out, of 3). Experiment 3: held-out contrastive AUC (naturalistic > HappyDB).

In [23]:
at_operating = stats[stats['eligible']].merge(operating['layer'].reset_index(), on=['model', 'layer'])

comparison = DataFrame({
    'operating layer': operating['layer'].astype(int).astype(str) + ' / ' + operating['blocks'].astype(int).astype(str),
    'depth': operating['rel_depth'].map('{:.0%}'.format),
    'agreeing metrics': operating['n_peaks'].astype(int).astype(str) + ' / ' + str(n_metrics),
    'max gated pseudo-F': at_operating.groupby('model')['F'].max().reindex(operating.index).round(2),
    'symptom agreement with clinician': [f"{downstream[mdl]['alignment']} / 3" if mdl in downstream else ''
                            for mdl in operating.index],
    'Gram condition number': [downstream[mdl]['gram_cond'] if mdl in downstream else None
                              for mdl in operating.index],
    'contrastive AUC': [round(downstream[mdl]['auc'], 3) if 'auc' in downstream.get(mdl, {}) else None
                        for mdl in operating.index],
}, index=operating.index).rename(index=display_name)
comparison.index.name = 'model'
comparison

,operating layer,depth,agreeing metrics,max gated pseudo-F,symptom agreement with clinician,Gram condition number,contrastive AUC
model,,,,,,,
Gemma-3-27B-PT,21 / 62,34%,3 / 8,6.91,3 / 3,14624.518478,0.789
Gemma-3-27B-IT,21 / 62,34%,5 / 8,8.40,3 / 3,12484.653396,0.730
Gemma-3-12B-PT,13 / 48,27%,3 / 8,5.91,3 / 3,8959.238961,0.959
Gemma-3-12B-IT,14 / 48,29%,2 / 8,8.03,3 / 3,18143.616996,0.965
Gemma-3-4B-PT,20 / 34,59%,3 / 8,6.55,3 / 3,12491.726341,0.945
Gemma-3-4B-IT,20 / 34,59%,4 / 8,8.45,3 / 3,11619.510286,0.953
Mistral-24B-PT,8 / 40,20%,5 / 8,5.84,3 / 3,35.490359,0.943
Mistral-24B-IT,8 / 40,20%,5 / 8,6.16,3 / 3,37.149830,0.946
OLMo-2-32B-PT,16 / 64,25%,3 / 8,5.41,3 / 3,33.503474,0.945


In [24]:
# The comparison above as an interactive table.
TABLE_HTML = f'{TMLR_DIR}/82.multimodel_comparison_table.html'

fig_comparison = go.Figure(go.Table(
    columnwidth=[1.5, 1.2, 0.85, 0.85, 0.85, 0.85, 0.85],
    header=dict(values=['<b>Model</b>', '<b>Operating layer</b><br>layer / blocks · depth',
                        '<b>Agreeing</b><br>metrics', '<b>Max gated</b><br>pseudo-F',
                        '<b>Clinician</b><br>alignment', '<b>Gram</b><br>κ(G)', '<b>Contrastive</b><br>AUC'],
                fill_color='#f3f3f3', line_color='#cccccc', align='center', font=TICK, height=48),
    cells=dict(values=[comparison.index,
                       comparison['operating layer'] + ' · ' + comparison['depth'],
                       comparison['agreeing metrics'],
                       comparison['max gated pseudo-F'].map('{:.2f}'.format),
                       comparison['symptom agreement with clinician'],
                       comparison['Gram condition number'].map(lambda v: '' if pd.isna(v) else f'{v:,.1f}'),
                       comparison['contrastive AUC'].map(lambda v: '' if pd.isna(v) else f'{v:.3f}')],
               fill_color='white', line_color='#e5e5e5', align=['left'] + ['center'] * 6, font=TICK, height=30),
))
fig_comparison.update_layout(margin=dict(l=0, r=0, t=0, b=0), height=48 + 30 * len(comparison) + 30)
fig_comparison.show()

## Export the multi-model figures

Writes the per-model tables and figures to TMLR_DIR. Off by default: adjust them above
(the dropdowns work inline), then set `EXPORT_MODELS_TMLR = True` and run this cell.

In [25]:
EXPORT_MODELS_TMLR = True
if EXPORT_MODELS_TMLR:
    for fig, path in [(fig_table_permanova, TABLE_PERMANOVA_HTML),
                      (fig_coef_abc, FIG_COEF_ABC_HTML), (fig_coef_rank, FIG_COEF_RANK_HTML),
                      (fig_3d, FIG_MODELS_3D_HTML), (fig_auc, FIG_MODELS_AUC_HTML), (fig_comparison, TABLE_HTML)]:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        (write_tmlr_dropdown_html if fig.layout.updatemenus else write_tmlr_html)(fig, path)
        print("wrote", path)

wrote tmlr-beyond-pdf-author-kit/submission_folder/assets/html/submission/82.multimodel_table_permanova.html
wrote tmlr-beyond-pdf-author-kit/submission_folder/assets/html/submission/82.multimodel_coefficient_abc.html
wrote tmlr-beyond-pdf-author-kit/submission_folder/assets/html/submission/82.multimodel_coefficient_rank.html
wrote tmlr-beyond-pdf-author-kit/submission_folder/assets/html/submission/82.multimodel_3d.html
wrote tmlr-beyond-pdf-author-kit/submission_folder/assets/html/submission/82.multimodel_auc.html
wrote tmlr-beyond-pdf-author-kit/submission_folder/assets/html/submission/82.multimodel_comparison_table.html
